## Evolution Analysis

In [1]:
from datetime import datetime
date = datetime.now()
formatted_date = date.strftime("%B %d, %Y")
print(formatted_date)

April 04, 2025


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
userdata.get('HF_TOKEN')

# Set up the current working directory within the Google Drive
%cd /content/drive/My\ Drive/Colab\ Notebooks/LLM/sped_biblio/evolution

Mounted at /content/drive
/content/drive/My Drive/Colab Notebooks/LLM/sped_biblio/evolution


In [3]:
# !pip install dill qgrid

# !pip install spacy
# !python -m spacy download en_core_web_md
# !pip install --upgrade -q plotly
# !pip install -q pandas==2.2.2 numpy==1.26.4

In [4]:
import re
import ast
import warnings
from collections import defaultdict
import pickle
from pickle import UnpicklingError

# Data Manipulation
import numpy as np
import pandas as pd
import requests
import time
from collections import defaultdict

# Natural Language Processing
import requests
import re
import nltk
import spacy
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

# Network Analysis
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import networkx as nx

# Progress Bar
from tqdm import tqdm

# Display HTML
from IPython.display import IFrame

#### New Word Counts

In [5]:
df = pd.read_excel("files/df.xlsx")

In [6]:
def clean_literal(s):
    s = s.strip().replace('\n', ' ')
    if not s.startswith('['):
        s = '[' + s
    if not s.endswith(']'):
        s = s + ']'
    return s

def safe_eval_if_str(val):
    if isinstance(val, str):
        try:
            cleaned = clean_literal(val)
            return ast.literal_eval(cleaned)
        except Exception as e:
            val = val.strip("[]")
            return [item.strip(" '") for item in val.split(",")]
    return val

In [7]:
df['unigrams'] = df['unigrams'].apply(safe_eval_if_str)
df['bigrams']  = df['bigrams'].apply(safe_eval_if_str)
df['trigrams'] = df['trigrams'].apply(safe_eval_if_str)

In [8]:
df_unigrams = df[['UT', 'Year', 'Topic', 'unigrams']].copy()
df_unigrams['token_count'] = df_unigrams['unigrams'].apply(len)
df_unigrams = df_unigrams.explode('unigrams').rename(columns={'unigrams': 'ngram'})
df_unigrams['ngram_type'] = 'unigrams'

df_bigrams = df[['UT', 'Year', 'Topic', 'bigrams']].copy()
df_bigrams['token_count'] = df_bigrams['bigrams'].apply(len)
df_bigrams = df_bigrams.explode('bigrams').rename(columns={'bigrams': 'ngram'})
df_bigrams['ngram_type'] = 'bigrams'

df_trigrams = df[['UT', 'Year', 'Topic', 'trigrams']].copy()
df_trigrams['token_count'] = df_trigrams['trigrams'].apply(len)
df_trigrams = df_trigrams.explode('trigrams').rename(columns={'trigrams': 'ngram'})
df_trigrams['ngram_type'] = 'trigrams'

ngrams_concat = pd.concat([df_unigrams, df_bigrams, df_trigrams], ignore_index=True)

In [9]:
ngrams_concat.to_pickle("results/ngrams_concat.pkl")

In [10]:
unigrams_df = ngrams_concat[ngrams_concat['ngram_type'].str.lower() == 'unigrams'].copy()
unigrams_df['Decade'] = unigrams_df['Year'].apply(lambda y: f"{int(y)//10*10}s")

def process_tokens(tokens):
    cnt = tokens.value_counts()
    once = cnt[cnt == 1].index.tolist()
    multi = cnt[cnt > 1].index.tolist()
    all_tok = cnt.index.tolist()

    freq_df = pd.DataFrame({'frequency': cnt})
    desc_stats = freq_df['frequency'].describe()
    skew_val = freq_df['frequency'].skew()
    kurt_val = freq_df['frequency'].kurt()

    return pd.DataFrame([{
        'All Tokens': all_tok,
        'Distinct Token Count': len(all_tok),
        'Tokens Appearing Once': once,
        'Count (Once)': len(once),
        'Tokens Appearing >1 Time': multi,
        'Count (>1 Time)': len(multi),
        'Total Token Count': cnt.sum(),
        'Verification (Once + Multi)': len(once) + len(multi),
        'Min Frequency': desc_stats['min'],
        '25% Frequency': desc_stats['25%'],
        'Median Frequency': desc_stats['50%'],
        '75% Frequency': desc_stats['75%'],
        'Max Frequency': desc_stats['max'],
        'Mean Frequency': desc_stats['mean'],
        'Std Frequency': desc_stats['std'],
        'Skew Frequency': skew_val,
        'Kurt Frequency': kurt_val,
    }])

unigrams_group_df = []
# for (topic, decade), sub_df in unigrams_df.groupby(['Topic', 'Decade']):
for decade, sub_df in unigrams_df.groupby('Decade'):
    token_stats = process_tokens(sub_df['ngram'])
    # token_stats['Topic'] = topic
    token_stats['Decade'] = decade
    unigrams_group_df.append(token_stats)

unigrams_concat = pd.concat(unigrams_group_df, ignore_index=True)

def calculate_new_word_count(df):
    grouped = df.groupby(['Topic', 'Year'])['ngram'].apply(set).reset_index()
    count_df = []
    for topic, group in grouped.groupby('Topic'):
        group = group.sort_values('Year')
        previous_tokens = set()
        for _, row in group.iterrows():
            current_tokens = row['ngram']
            new_tokens = current_tokens - previous_tokens
            new_word_count = len(new_tokens)
            count_df.append({
                'Topic': topic,
                'Year': row['Year'],
                'new_word_count': new_word_count
            })
            previous_tokens.update(current_tokens)
    return pd.DataFrame(count_df)

new_word_count_df = calculate_new_word_count(unigrams_df)

In [11]:
unigrams_concat.to_excel("results/unigrams_concat.xlsx", index=False)
new_word_count_df.to_excel("results/new_word_count_df.xlsx", index=False)

In [15]:
from nbconvert import HTMLExporter
import nbformat

notebook_path = 'index.ipynb'
html_exporter = HTMLExporter()

with open(notebook_path, 'r', encoding='utf-8') as nb_file:
    notebook_content = nb_file.read()
    notebook = nbformat.reads(notebook_content, as_version=4)

if 'widgets' in notebook.metadata and 'application/vnd.jupyter.widget-state+json' in notebook.metadata['widgets']:
    if 'state' not in notebook.metadata['widgets']['application/vnd.jupyter.widget-state+json']:
        notebook.metadata['widgets']['application/vnd.jupyter.widget-state+json']['state'] = {}

html_output, _ = html_exporter.from_notebook_node(notebook)

with open('index.html', 'w', encoding='utf-8') as html_file:
    html_file.write(html_output)